# 02 — Parsing PDF → Chunking → Embedding → Indexing

**Tujuan:** mengubah korpus `ID_Regulations` (PDF) menjadi indeks vektor lokal yang siap di-*retrieve*.

Alur:

```
data/raw/ID_Regulations/*.pdf
        |  (1) parsing  (pypdf)
        v
data/processed/parsed_<tag>.jsonl      # teks per halaman + metadata
        |  (2) chunking (RecursiveCharacterTextSplitter)
        v
data/processed/chunks_<tag>.jsonl
        |  (3) embedding (sentence-transformers, lokal)
        v
vectorstore/                            # ChromaDB persisten (lokal, tidak di-commit)
```

> **Catatan eksperimen (RQ2):** tiga parameter di sel konfigurasi — `CHUNK_SIZE`, `CHUNK_OVERLAP`, `EMBED_MODEL` — **sengaja dijadikan variabel**. Ubah salah satunya, jalankan ulang notebook, dan hasilnya masuk ke `model/runs/` sebagai catatan percobaan (untuk melacak "angka ini dari run yang mana").

Prasyarat: `data/raw/ID_Regulations/` sudah terisi (lihat `notebook/01_unduh_dataset.ipynb`).

In [ ]:
import json
import re
from datetime import date
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm
from pypdf import PdfReader

# --- lokasi ---
RAW_DIR = Path("data/raw/ID_Regulations")
PROCESSED_DIR = Path("data/processed")
VECTOR_DIR = Path("vectorstore")
RUNS_DIR = Path("model/runs")
for d in (PROCESSED_DIR, VECTOR_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- parameter eksperimen (dibandingkan di RQ2) ---
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64
EMBED_MODEL = "BAAI/bge-m3"   # alternatif: "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
PARSE_LIMIT = 500             # batasi jumlah PDF saat uji; None = semua
TAG = f"cs{CHUNK_SIZE}_ov{CHUNK_OVERLAP}_{EMBED_MODEL.split('/')[-1]}"
COLLECTION = re.sub(r"[^a-zA-Z0-9_]", "_", f"rag_{TAG}")

print("Tag eksperimen :", TAG)
print("Collection     :", COLLECTION)
print("PDF tersedia   :", len(list(RAW_DIR.rglob('*.pdf'))))

## 1. Parsing PDF (pypdf)

Ekstraksi teks **per halaman**, sambil menyimpan metadata (`source`, `page`) supaya setiap jawaban bisa dilacak ke dokumen & halaman asalnya. **`data/raw/` tidak diubah** — hasilnya ditulis ke `data/processed/`.

In [ ]:
def extract_pages(pdf_path: Path):
    reader = PdfReader(str(pdf_path))
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = re.sub(r"\s+", " ", text).strip()
        if len(text) >= 40:            # buang halaman kosong / terlalu pendek
            yield {"source": pdf_path.name, "page": i + 1, "text": text}


pdf_files = sorted(RAW_DIR.rglob("*.pdf"))
if PARSE_LIMIT:
    pdf_files = pdf_files[:PARSE_LIMIT]
print("PDF akan diparsing:", len(pdf_files))

parsed = []
errors = 0
for p in tqdm(pdf_files, desc="Parsing PDF"):
    try:
        parsed.extend(extract_pages(p))
    except Exception as e:
        errors += 1
        if errors <= 5:
            print("GAGAL:", p.name, "->", e)

print("Halaman teks terkumpul:", len(parsed), "| gagal:", errors)

In [ ]:
parsed_file = PROCESSED_DIR / f"parsed_{TAG}.jsonl"
with parsed_file.open("w", encoding="utf-8") as f:
    for row in parsed:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

total_chars = sum(len(r["text"]) for r in parsed)
print("Disimpan        :", parsed_file.resolve())
print("Total karakter  :", f"{total_chars:,}")

## 2. Chunking

Memotong tiap halaman menjadi potongan berukuran `CHUNK_SIZE` karakter dengan `CHUNK_OVERLAP` tumpang tindih. Ubah angkanya untuk membandingkan konfigurasi (RQ2).

In [ ]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:                                   # kompatibilitas langchain lama
    from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

chunks = []
for row in parsed:
    for j, part in enumerate(splitter.split_text(row["text"])):
        chunks.append({
            "id": f'{row["source"]}_p{row["page"]}_c{j}',
            "text": part,
            "source": row["source"],
            "page": row["page"],
        })

print("Jumlah chunk :", len(chunks))
print("Contoh chunk :", chunks[0]["text"][:200] if chunks else "(kosong)")

In [ ]:
chunks_file = PROCESSED_DIR / f"chunks_{TAG}.jsonl"
with chunks_file.open("w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

avg_len = np.mean([len(c["text"]) for c in chunks]) if chunks else 0
print("Disimpan         :", chunks_file.resolve())
print("Rata-rata panjang:", round(float(avg_len), 1), "karakter/chunk")

## 3. Embedding (lokal, `sentence-transformers`)

Semua proses berjalan **di dalam mesin** — tidak ada dokumen yang dikirim ke API pihak ketiga (sejalan dengan UU PDP No. 27/2022 dan inti tema keamanan skripsi).

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBED_MODEL)
dim = model.get_sentence_embedding_dimension()
print("Model          :", EMBED_MODEL)
print("Dimensi vektor :", dim)

texts = [c["text"] for c in chunks]
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,     # memudahkan kemiripan kosinus
)
embeddings = np.asarray(embeddings, dtype="float32")
print("Bentuk matriks :", embeddings.shape)

## 4. Indexing ke ChromaDB (persisten, lokal)

Menyimpan vektor + teks + metadata ke `vectorstore/` (sudah di-`.gitignore`).

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=str(VECTOR_DIR))

# buat ulang collection agar bersih untuk tag ini
try:
    client.delete_collection(COLLECTION)
except Exception:
    pass
collection = client.create_collection(name=COLLECTION, metadata={"hnsw:space": "cosine"})

BATCH = 1000
for start in tqdm(range(0, len(chunks), BATCH), desc="Indexing"):
    batch = chunks[start:start + BATCH]
    collection.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        embeddings=embeddings[start:start + BATCH].tolist(),
        metadatas=[{"source": c["source"], "page": c["page"]} for c in batch],
    )

print("Selesai. Total vektor di collection:", collection.count())

## 5. Uji cepat (*smoke test*)

Mengambil 5 chunk terdekat untuk satu contoh pertanyaan — sekadar memastikan indeks berfungsi. Evaluasi akurasi sesungguhnya dilakukan di notebook evaluasi (RQ1–RQ2).

In [ ]:
query = "Berapa lama waktu kerja lembur dalam sehari?"
q_emb = model.encode([query], normalize_embeddings=True).tolist()
res = collection.query(query_embeddings=q_emb, n_results=5)

for i, (doc, meta, dist) in enumerate(
    zip(res["documents"][0], res["metadatas"][0], res["distances"][0]), 1
):
    print(f"{i}. [{meta['source']} hal.{meta['page']}] jarak={dist:.3f}")
    print("   ", doc[:200].replace("\n", " "), "...")

## 6. Catat konfigurasi run (ketertelusuran)

Menyimpan ringkasan percobaan ke `model/runs/` agar setiap angka di laporan bisa dilacak ke konfigurasi tertentu.

In [ ]:
run = {
    "date": date.today().isoformat(),
    "dataset": "ID_Regulations",
    "num_pdf": len(pdf_files),
    "num_pages": len(parsed),
    "num_chunks": len(chunks),
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embed_model": EMBED_MODEL,
    "embed_dim": int(dim),
    "collection": COLLECTION,
    "parsed_file": str(parsed_file),
    "chunks_file": str(chunks_file),
}
run_file = RUNS_DIR / f"run_{TAG}.json"
run_file.write_text(json.dumps(run, indent=2, ensure_ascii=False), encoding="utf-8")

print("Manifest run:", run_file.resolve())
print(json.dumps(run, indent=2, ensure_ascii=False))

## Langkah berikutnya

1. Ulangi dengan parameter lain (`CHUNK_SIZE` 256/1024, `EMBED_MODEL` berbeda) → bandingkan di RQ2.
2. Buat notebook evaluasi: tarik `eval/gold_qa.csv`, hitung **Recall@k / MRR / nDCG** dan **RAGAS** (RQ1–RQ2).
3. Buat notebook keamanan: index dokumen ber-*canary* dari `src/make_canary_docs.py`, jalankan prompt injeksi, hitung kebocoran & *refusal rate* (RQ3).
4. Catat latensi & puncak VRAM tiap run (RQ4).